# 📊 Báo Cáo Đánh Giá & Benchmark Thuật Toán Gợi Ý Sản Phẩm
Notebook này thực hiện kiểm thử, đánh giá hiệu năng và so sánh điểm số giữa hai thuật toán gợi ý:
1. **Collaborative Filtering (Lọc Cộng Tác Lai + User-Mean Centering)**: Áp dụng cho Trang Chủ.
2. **Content-Based Filtering (Lọc Dựa Trên Nội Dung - TF-IDF & Cosine Similarity)**: Áp dụng cho Trang Chi Tiết Sản Phẩm.

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from dotenv import load_dotenv

# Add backend to sys path
sys.path.append(os.path.abspath('../backend'))
from database import supabase

# Config plots
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.sans-serif'] = 'Segoe UI'
print('✓ Đã nạp thành công các thư viện kiểm thử!')

## 1. Nạp Dữ Liệu Tương Tác & Sản Phẩm từ CSDL Supabase

In [ ]:
# Fetch products and interactions
res_products = supabase.table('products').select('*').execute()
res_interactions = supabase.table('interactions').select('*').execute()

df_products = pd.DataFrame(res_products.data)
df_interactions = pd.DataFrame(res_interactions.data)

print(f"Tổng số sản phẩm trong DB: {len(df_products)}")
print(f"Tổng số tương tác trong DB: {len(df_interactions)}")
df_interactions.head()

## 2. Tính Toán Ma Trận Trọng Số Tương Tác (Hybrid Matrix Weighting)
- **Click xem chi tiết**: Trọng số = `1.0`
- **Thêm vào giỏ hàng**: Trọng số = `5.0`
- **Đánh giá sao**: Chuẩn hóa độ lệch Trung bình Người dùng $\text{Weight} = S_{u,i} - \mu_u$

In [ ]:
def calculate_user_means(df_int):
    ratings = df_int[(df_int['action'] == 'rating') | (df_int['rating_score'].notna())].copy()
    if ratings.empty:
        return {}
    ratings['score'] = ratings.apply(lambda r: float(r['rating_score']) if pd.notna(r.get('rating_score')) else 5.0, axis=1)
    return ratings.groupby('user_id')['score'].mean().to_dict()

user_means = calculate_user_means(df_interactions)

def get_interaction_weight(row):
    action = row.get('action', 'click')
    rating = row.get('rating_score')
    u_id = str(row.get('user_id'))
    
    if action == 'rating' or (rating is not None and not pd.isna(rating)):
        score = float(rating) if (rating is not None and not pd.isna(rating)) else 5.0
        u_mean = user_means.get(u_id, 3.0)
        return score - u_mean
    elif action == 'add_to_cart':
        return 5.0
    elif action == 'click':
        return 1.0
    return 1.0

df_interactions['weight'] = df_interactions.apply(get_interaction_weight, axis=1)
user_item_matrix = df_interactions.groupby(['user_id', 'product_id'])['weight'].sum().unstack(fill_value=0)

print("Kích thước Ma trận User-Item:", user_item_matrix.shape)
user_item_matrix.iloc[:5, :5]

## 3. Định Nghĩa Các Chỉ Số Đánh Giá (Evaluation Metrics)
- **Precision@K**: Tỷ lệ sản phẩm gợi ý đúng nhu cầu trên tổng số K sản phẩm được gợi ý.
- **Recall@K**: Tỷ lệ sản phẩm gợi ý đúng nhu cầu trên tổng số sản phẩm thực tế người dùng yêu thích.
- **F1-Score@K**: Trung bình điều hòa giữa Precision@K và Recall@K.
- **MAP@K (Mean Average Precision)**: Đo lường độ chính xác theo thứ tự xếp hạng của danh sách gợi ý.

In [ ]:
def precision_at_k(actual, recommended, k=10):
    rec_k = recommended[:k]
    relevant = set(actual).intersection(set(rec_k))
    return len(relevant) / k if k > 0 else 0.0

def recall_at_k(actual, recommended, k=10):
    rec_k = recommended[:k]
    relevant = set(actual).intersection(set(rec_k))
    return len(relevant) / len(actual) if len(actual) > 0 else 0.0

def f1_score_at_k(p, r):
    return (2 * p * r) / (p + r) if (p + r) > 0 else 0.0

def map_at_k(actual, recommended, k=10):
    rec_k = recommended[:k]
    score = 0.0
    num_hits = 0.0
    for i, p in enumerate(rec_k):
        if p in actual:
            num_hits += 1.0
            score += num_hits / (i + 1.0)
    return score / min(len(actual), k) if len(actual) > 0 else 0.0

## 4. Chạy Benchmark & Đánh Giá Điểm Số Các Thuật Toán

In [ ]:
from recsys.collaborative import get_collaborative_recommendations
from recsys.content_based import get_content_based_recommendations

test_users = list(user_item_matrix.index[:10])
k_values = [5, 10, 20]
results = []

for k in k_values:
    precisions, recalls, f1s, maps = [], [], [], []
    for uid in test_users:
        user_actual = list(user_item_matrix.loc[uid][user_item_matrix.loc[uid] > 0].index)
        if not user_actual:
            continue
        recs = get_collaborative_recommendations(uid, top_n=50)
        rec_pids = [p['_id'] for p in recs]
        
        p = precision_at_k(user_actual, rec_pids, k=k)
        r = recall_at_k(user_actual, rec_pids, k=k)
        f1 = f1_score_at_k(p, r)
        m = map_at_k(user_actual, rec_pids, k=k)
        
        precisions.append(p)
        recalls.append(r)
        f1s.append(f1)
        maps.append(m)
        
    results.append({
        'K': k,
        'Precision@K': np.mean(precisions),
        'Recall@K': np.mean(recalls),
        'F1-Score@K': np.mean(f1s),
        'MAP@K': np.mean(maps)
    })

df_metrics = pd.DataFrame(results)
df_metrics

## 5. Trực Quan Hóa Kết Quả Benchmark

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Plot Precision & Recall @ K
ax[0].plot(df_metrics['K'], df_metrics['Precision@K'], marker='o', linewidth=2.5, color='#4f46e5', label='Precision@K')
ax[0].plot(df_metrics['K'], df_metrics['Recall@K'], marker='s', linewidth=2.5, color='#10b981', label='Recall@K')
ax[0].set_title('Precision@K & Recall@K theo K', fontsize=12, fontweight='bold')
ax[0].set_xlabel('K (Số sản phẩm gợi ý)')
ax[0].set_ylabel('Điểm số')
ax[0].legend()

# Plot MAP@K & F1-Score@K
ax[1].bar(df_metrics['K'] - 0.8, df_metrics['MAP@K'], width=1.5, color='#8b5cf6', label='MAP@K')
ax[1].bar(df_metrics['K'] + 0.8, df_metrics['F1-Score@K'], width=1.5, color='#f59e0b', label='F1-Score@K')
ax[1].set_title('MAP@K & F1-Score@K', fontsize=12, fontweight='bold')
ax[1].set_xlabel('K (Số sản phẩm gợi ý)')
ax[1].set_ylabel('Điểm số')
ax[1].legend()

plt.tight_layout()
plt.show()